In [ ]:
#if you need download
%pip install pdfplumber pandas openpyxl


import re
import pdfplumber
import pandas as pd
import os



def extract_report(pdf_path):

    # ----------------------------------------
    # Part 1: Extract mean ± sd values
    # ----------------------------------------

    means = []
    sds = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""

            # Match formats such as:
            # 64.0±1.6
            # 64.0 ± 1.6
            matches = re.findall(
                r'(-?\d+\.?\d*)\s*±\s*(-?\d+\.?\d*)',
                text
            )

            for m, s in matches:
                means.append(float(m))
                sds.append(float(s))

    df = pd.DataFrame({
        'mean': means,
        'sd': sds
    })


    # ----------------------------------------
    # Part 2: Extract left/right values
    # ----------------------------------------

    keywords = [
        "最大力值1, N",
        "时间最大力值1, %",
        "最大力值2, N",
        "时间最大力值2, %",
    ]


    def first_num_after_side(line: str, side: str):

        if not line:
            return None

        pattern = rf"{side}\s+(-?\d+(?:[.,]\d*)?)"
        m = re.search(pattern, line)

        if not m:
            return None

        return float(
            m.group(1).replace(",", ".")
        )


    results = {
        kw: []
        for kw in keywords
    }


    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            text = page.extract_text() or ""
            lines = text.split("\n")

            for i, line in enumerate(lines):

                for kw in keywords:

                    if kw in line and not results[kw]:

                        L_line = (
                            lines[i - 1]
                            if i - 1 >= 0
                            else ""
                        )

                        R_line = (
                            lines[i + 1]
                            if i + 1 < len(lines)
                            else ""
                        )

                        L_val = first_num_after_side(
                            L_line,
                            "L"
                        )

                        R_val = first_num_after_side(
                            R_line,
                            "R"
                        )

                        vals = []

                        if L_val is not None:
                            vals.append(L_val)

                        if R_val is not None:
                            vals.append(R_val)

                        results[kw] = vals


    b = results


    # ----------------------------------------
    # Part 3: Extract max gait line velocity
    # ----------------------------------------

    max_gait_val = None

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            text = page.extract_text() or ""

            for line in text.split("\n"):

                if "Max gait line velocity" in line:

                    nums = re.findall(
                        r"-?\d+(?:[.,]\d*)?",
                        line
                    )

                    if nums:

                        max_gait_val = float(
                            nums[0].replace(",", ".")
                        )

                    break


    a = max_gait_val


    # ----------------------------------------
    # Part 4: Combine all extracted results
    # ----------------------------------------

    df_all = df.copy()

    if 'L' not in df_all.columns:
        df_all['L'] = None

    if 'R' not in df_all.columns:
        df_all['R'] = None


    df_a = pd.DataFrame(
        {
            "mean": [None],
            "sd": [None],
            "L": [a],
            "R": [None],
        },
        index=["Max gait line velocity"],
    )


    df_b = pd.DataFrame(
        b,
        index=["L", "R"]
    ).T

    df_b["mean"] = None
    df_b["sd"] = None

    df_b = df_b[
        ["mean", "sd", "L", "R"]
    ]


    df_out = pd.concat([
        df_all,
        df_a,
        df_b
    ])


    # ----------------------------------------
    # Part 5: Export to Excel
    # ----------------------------------------

    excel_name = (
        os.path.splitext(
            os.path.basename(pdf_path)
        )[0]
        + ".xlsx"
    )

    df_out.to_excel(
        excel_name,
        index=True
    )

    print(
        f"Extraction completed. "
        f"Output saved as: {excel_name}"
    )

    return df_out


# ----------------------------------------
# Example
# ----------------------------------------

df_result = extract_report(
    "58bpmDT.pdf"
)

print(df_result)

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 6.6/6.6 MB 36.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 38.5 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 3.9/3.9 MB 38.5 MB/s  0:00:00

   ---------------------------------------- 0/8 [pypdfium2]
   --------------- ------------------------ 3/8 [openpyxl]
   --------------- ------------------------ 3/8 [openpyxl]
   --------------- ------------------------ 3/8 [openpyxl]
   -------------------- ------------------- 4/8 [cffi]
   ------------------------- -------------- 5/8 [cryptography]
   ------------------------------ --------- 6/8 [pdfminer.six]
   ------------------------------ --------- 6/8 [pdfminer.six]
   ----------------------------------- ---- 7/8 [pdfplumber]
   -------------------------